In [ ]:
"""
ERA5 Daily Total Precipitation Processor

Core functions to convert ERA5 hourly precipitation (meters) to daily totals (millimeters).
Caller handles file paths, date ranges, and I/O operations.
"""

import xarray as xr
from datetime import datetime, timedelta
from pathlib import Path
from typing import Union


def compute_daily_precipitation(
    file_path: Union[str, Path],
    date_label: str,
    mm_conversion: float = 1000.0
) -> xr.Dataset:
    """
    Compute daily total precipitation from ERA5 hourly file.
    
    Parameters
    ----------
    file_path : str or Path
        Path to NetCDF file containing 'tp' variable in meters
    date_label : str
        Date identifier (e.g., '20200115') for output dimension
    mm_conversion : float
        Multiplier to convert meters to millimeters (default: 1000.0)
    
    Returns
    -------
    xr.Dataset
        Dataset with daily total precipitation in millimeters
    """
    ds = xr.open_dataset(file_path)
    
    # Convert to mm and compute daily sum
    precip_mm = ds['tp'] * mm_conversion
    
    # Auto-detect time dimension (ERA5 uses 'valid_time' or 'time')
    time_dim = 'valid_time' if 'valid_time' in ds.dims else 'time'
    daily_sum = precip_mm.sum(dim=time_dim).expand_dims(date=[date_label])
    
    # Package result
    result = daily_sum.to_dataset(name='tp')
    result['tp'].attrs['units'] = 'mm'
    return result


def process_precipitation_range(
    base_directory: Union[str, Path],
    start_date: datetime,
    end_date: datetime,
    mm_conversion: float = 1000.0
) -> xr.Dataset:
    """
    Process consecutive days to compute daily total precipitation.
    
    Parameters
    ----------
    base_directory : str or Path
        Base path containing yearly subdirectories (e.g., '.../2020/')
    start_date : datetime
        First date to process (inclusive)
    end_date : datetime
        Last date to process (exclusive)
    mm_conversion : float
        Multiplier to convert meters to millimeters
    
    Returns
    -------
    xr.Dataset
        Concatenated daily precipitation totals for date range
    """
    base_dir = Path(base_directory)
    date_range = [
        start_date + timedelta(days=i)
        for i in range((end_date - start_date).days)
    ]
    
    # Process each day
    daily_datasets = [
        compute_daily_precipitation(
            file_path=base_dir / str(dt.year) / f"ERA5_P_{dt.strftime('%Y%m%d')}.nc",
            date_label=dt.strftime('%Y%m%d'),
            mm_conversion=mm_conversion
        )
        for dt in date_range
    ]
    
    # Combine results
    combined = xr.concat(daily_datasets, dim='date')
    combined.attrs.update({
        'processing': 'Daily total precipitation',
        'source': 'ERA5 Reanalysis',
        'conversion': 'Meters to millimeters (×1000)'
    })
    return combined